# Exploração inicial dos dados

Primeira leitura das tabelas do Olist: tamanhos, chaves, período e
recompra bruta. Nada aqui gera arquivos; os achados viraram decisões em
[docs/decisions.md](../docs/decisions.md).

## Tamanho das tabelas

In [ ]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")   # ajuste se necessário

t = {p.stem.replace("olist_", "").replace("_dataset", ""): pd.read_csv(p)
     for p in RAW.glob("*.csv")}

for nome, df in t.items():
    print(nome, df.shape)
    print(list(df.columns), "\n")

## Chave do cliente e recompra bruta

In [ ]:
c, o = t["customers"], t["orders"]
print(c["customer_id"].nunique(), c["customer_unique_id"].nunique())
print(o["order_id"].is_unique)

base = o.merge(c, on="customer_id")
print((base.groupby("customer_unique_id").size() > 1).mean())

## Linhas por pedido em cada tabela

In [ ]:
print(t["order_items"].groupby("order_id").size().describe())
print(t["order_payments"].groupby("order_id").size().describe())
print((t["order_reviews"].groupby("order_id").size() > 1).sum())

## Teste do fan-out

In [ ]:
itens = t["order_items"]
certo = itens["price"].sum()
errado = itens.merge(t["order_payments"], on="order_id")["price"].sum()
print(certo, errado)

## Período coberto

In [ ]:
o["order_purchase_timestamp"] = pd.to_datetime(o["order_purchase_timestamp"])
print(o.groupby(o["order_purchase_timestamp"].dt.to_period("M")).size())
print(o["order_purchase_timestamp"].max())

## Dias até o segundo pedido

In [ ]:
b = base.copy()
b["order_purchase_timestamp"] = pd.to_datetime(b["order_purchase_timestamp"])
b = b.sort_values(["customer_unique_id", "order_purchase_timestamp"])
b["n"] = b.groupby("customer_unique_id").cumcount()

primeiro = b[b["n"] == 0].set_index("customer_unique_id")["order_purchase_timestamp"]
segundo = b[b["n"] == 1].set_index("customer_unique_id")["order_purchase_timestamp"]
dias = (segundo - primeiro.loc[segundo.index]).dt.days

print(dias.describe())
print("mesmo dia:", (dias == 0).mean())
print("até 180 dias:", (dias <= 180).mean())

## Recompra sem os pedidos do mesmo dia

In [ ]:
b["data"] = b["order_purchase_timestamp"].dt.normalize()
primeira = b.groupby("customer_unique_id")["data"].transform("min")
b["dias_desde_primeiro"] = (b["data"] - primeira).dt.days

total = b["customer_unique_id"].nunique()
rec = b.loc[b["dias_desde_primeiro"] > 0, "customer_unique_id"].nunique()
rec180 = b.loc[b["dias_desde_primeiro"].between(1, 180), "customer_unique_id"].nunique()
print(total, rec, rec180, rec / total)

## Status dos pedidos

In [ ]:
o["order_status"].value_counts()

## Pedidos sem avaliação

In [ ]:
sem_av = ~t["orders"]["order_id"].isin(t["order_reviews"]["order_id"])
print(sem_av.sum(), sem_av.mean())

## Janela de observação

In [ ]:
print(o.loc[o["order_purchase_timestamp"] < "2018-09-01", "order_purchase_timestamp"].max())

FIM = pd.Timestamp("2018-08-31 16:13:44")
LIMITE_PRIMEIRO_PEDIDO = FIM - pd.Timedelta(days=180)
print(FIM, LIMITE_PRIMEIRO_PEDIDO)

## Pedidos entregues antes de 2017

In [ ]:
p2016 = o[(o["order_status"] == "delivered") & (o["order_purchase_timestamp"] < "2017-01-01")]
print(len(p2016))